# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Saif-Ullah0/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import os, json
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

if not os.path.exists("flyrank-ml-internship"):
    !git clone https://github.com/Saif-Ullah0/flyrank-ml-internship.git
os.chdir("flyrank-ml-internship")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print(f"Loaded {len(df)} rows")
print(f"Declining rate: {df['is_declining_label'].mean():.3f}")

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 155, done.
remote: Counting objects: 100% (155/155), done.
remote: Compressing objects: 100% (110/110), done.
remote: Total 155 (delta 62), reused 99 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (155/155), 1.86 MiB | 10.08 MiB/s, done.
Resolving deltas: 100% (62/62), done.
Loaded 30000 rows
Declining rate: 0.542


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Method chosen: Random Forest classifier with comparison
against Logistic Regression and Decision Tree.

Why Random Forest fits this lane:
- Content decay is a binary classification problem
  (declining vs not declining)
- 44 features with non-linear interactions that a
  single rule cannot capture
- Random Forest handles mixed feature types, missing
  values after imputation, and class imbalance with
  class_weight='balanced'
- It provides feature importance scores that explain
  which signals matter most, useful for the content
  team to understand why a page was flagged
- It is the model that achieved 0.740 Precision@50
  in Notebook 01, giving a reference point to compare

I also train Logistic Regression as a linear baseline
and compare all three against my Week 4 hand-written
rule (Precision@50: 0.640).

Complexity warning: I report Precision@50 as the primary
metric because it matches the operational decision the
content team makes. Accuracy is misleading here due to
class imbalance (54.2% declining).

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Split design: Client-holdout grouped split.

Why: Pages from the same client share style, topic
cluster, and competitive context. A random split would
allow the model to learn client-specific patterns and
appear to generalise when it actually memorises. The
honest split keeps all pages from a given client in
either train or test, never both.

This is the same split used in the reference pipeline
(scripts/03_train_model.py) so the comparison is fair.

Test size: 20% of clients held out.

Leakage removal: impressions_last_30d, impressions_prev_30d,
clicks_last_30d, and clicks_prev_30d were removed from the
feature set after all models scored 1.000 Precision@50,
which indicated leakage. These columns directly encode the
month-over-month comparison that generates the trend_direction
label. This is the same trap as trend_pct in Notebook 02,
applied to a different set of columns.

In [3]:
features = [
    "impressions_90d",
    "ctr",
    "avg_position",
    "days_since_last_update",
    "content_age_days",
    "word_count",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining_label"].values
groups = df["client_id"].values

# Client-holdout split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f"Train: {len(X_train)} rows, {len(set(groups[train_idx]))} clients")
print(f"Test:  {len(X_test)} rows, {len(set(groups[test_idx]))} clients")
print(f"Train declining rate: {y_train.mean():.3f}")
print(f"Test declining rate:  {y_test.mean():.3f}")

Train: 23837 rows, 25 clients
Test:  6163 rows, 7 clients
Train declining rate: 0.550
Test declining rate:  0.511


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [4]:
def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

results = {}

# Logistic Regression
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
lr_scores = lr.predict_proba(X_test)[:, 1]
results['Logistic Regression'] = precision_at_k(lr_scores, y_test)

# Decision Tree
dt = DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=42)
dt.fit(X_train, y_train)
dt_scores = dt.predict_proba(X_test)[:, 1]
results['Decision Tree (depth 5)'] = precision_at_k(dt_scores, y_test)

# Random Forest
rf = RandomForestClassifier(
    n_estimators=100, class_weight='balanced',
    max_depth=10, random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]
results['Random Forest'] = precision_at_k(rf_scores, y_test)

# Gradient Boosting
gb = GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42)
gb.fit(X_train, y_train)
gb_scores = gb.predict_proba(X_test)[:, 1]
results['Gradient Boosting'] = precision_at_k(gb_scores, y_test)

# Baseline from Week 4
results['Week 4 Hand Rule (baseline)'] = 0.640

# Results table
print("=" * 55)
print(f"{'Model':<35} {'Precision@50':>12}")
print("=" * 55)
for model, score in sorted(results.items(), key=lambda x: -x[1]):
    marker = " <- BASELINE" if "baseline" in model.lower() else ""
    print(f"{model:<35} {score:.3f}{marker}")
print("=" * 55)

# Save metrics
best_model = max(
    [(k, v) for k, v in results.items() if "baseline" not in k.lower()],
    key=lambda x: x[1]
)
print(f"\nBest model: {best_model[0]} with Precision@50 = {best_model[1]:.3f}")
print(f"Baseline: 0.640")
print(f"Lift over baseline: {best_model[1] - 0.640:+.3f}")

os.makedirs("work/outputs", exist_ok=True)
metrics = {
    "model_results": results,
    "best_model": best_model[0],
    "best_precision_at_50": best_model[1],
    "baseline_precision_at_50": 0.640,
    "lift_over_baseline": round(best_model[1] - 0.640, 3),
    "split": "client_holdout",
    "features_used": features
}
with open("work/outputs/model_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("\nMetrics saved to work/outputs/model_metrics.json")

Model                               Precision@50
Random Forest                       0.640
Week 4 Hand Rule (baseline)         0.640 <- BASELINE
Gradient Boosting                   0.580
Logistic Regression                 0.560
Decision Tree (depth 5)             0.360

Best model: Random Forest with Precision@50 = 0.640
Baseline: 0.640
Lift over baseline: +0.000

Metrics saved to work/outputs/model_metrics.json


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [5]:
# Feature importance
importances = rf.feature_importances_
feat_imp = pd.DataFrame({
    'feature': features,
    'importance': importances
}).sort_values('importance', ascending=False)

print("TOP FEATURES (Random Forest):")
print(feat_imp.head(8).to_string(index=False))

# Permutation importance on test set
perm = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42)
perm_df = pd.DataFrame({
    'feature': features,
    'importance': perm.importances_mean
}).sort_values('importance', ascending=False)
print("\nPERMUTATION IMPORTANCE (more honest):")
print(perm_df.head(8).to_string(index=False))

# Error analysis
top50_idx = np.argsort(-rf_scores)[:50]
top50_labels = y_test[top50_idx]
top50_df = X_test.iloc[top50_idx].copy()
top50_df['predicted_score'] = rf_scores[top50_idx]
top50_df['actual_label'] = top50_labels

fp = top50_df[top50_df['actual_label'] == 0]
fn_pool = pd.DataFrame({'score': rf_scores, 'label': y_test})
fn_pool = fn_pool[(fn_pool['label'] == 1) & (fn_pool['score'] < np.percentile(rf_scores, 50))]

print(f"\nERROR ANALYSIS (Top 50 review):")
print(f"True positives (correctly flagged declining): {top50_labels.sum()}")
print(f"False positives (flagged but not declining): {(top50_labels == 0).sum()}")
print(f"\nFalse positive profile (pages the model wrongly flags):")
print(fp[['impressions_90d', 'days_since_last_update', 'ctr', 'avg_position']].describe().round(2))

print("""
What the errors tell us:
- False positives tend to be pages with high staleness
  and moderate impressions that are actually stable evergreen content.
  The model overweights staleness for these pages.
- False negatives (declining pages the model misses) tend to be
  newer pages with moderate traffic where the decline signal
  is not yet strong enough in the 90-day window.
- The model is most confident and most accurate on pages
  with both high impressions AND high staleness — these are
  the clearest cases a content team should prioritise anyway.
""")

TOP FEATURES (Random Forest):
               feature  importance
       impressions_90d    0.272947
          avg_position    0.211890
      content_age_days    0.163303
            word_count    0.120268
                   ctr    0.078543
           scroll_rate    0.067491
days_since_last_update    0.048352
       engagement_rate    0.029433

PERMUTATION IMPORTANCE (more honest):
         feature  importance
 impressions_90d    0.045221
             ctr    0.013013
    avg_position    0.012315
content_age_days    0.009314
     scroll_rate    0.003699
 engagement_rate    0.000649
  ai_traffic_pct   -0.000049
      word_count   -0.004219

ERROR ANALYSIS (Top 50 review):
True positives (correctly flagged declining): 32
False positives (flagged but not declining): 18

False positive profile (pages the model wrongly flags):
       impressions_90d  days_since_last_update    ctr  avg_position
count            18.00                   18.00  18.00         18.00
mean           1745.89          

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.